In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    StringType,
    StructField,
    StructType,
)

# Initialize Spark session (Colab-compatible)
spark = SparkSession.builder.appName("MetadataControlTable").getOrCreate()

# ---------------------------------------
# STEP 1: Drop table if exists
# ---------------------------------------
spark.sql("DROP TABLE IF EXISTS control_table")

# ---------------------------------------
# STEP 2: Create config DataFrame schema
# ---------------------------------------
schema = StructType(
    [
        StructField("source_name", StringType(), True),
        StructField("entity_name", StringType(), True),
        StructField("source_type", StringType(), True),
        StructField("source_path", StringType(), True),
        StructField("load_type", StringType(), True),
        StructField("watermark_column", StringType(), True),
        StructField("last_processed_value", StringType(), True),
        StructField("mapping_config_path", StringType(), True),
        StructField("transformation_module", StringType(), True),
        StructField("schedule_type", StringType(), True),
        StructField("active_flag", BooleanType(), True),
        StructField("config_version", StringType(), True),
        StructField("data_quality_threshold", DoubleType(), True),
    ]
)

# ---------------------------------------
# STEP 3: Insert sample records
# ---------------------------------------
records = [
    ("carrier_crm", "sales_transactions", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_sales.json", None, "daily", True, "v1", 0.9),
    ("carrier_crm", "customers", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_customers.json", None, "daily", True, "v1", 0.9),
    ("carrier_crm", "policies", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_policies.json", None, "daily", True, "v1", 0.9),
    ("oas_db", "sales_transactions", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_sales.json", None, "hourly", True, "v1", 0.9),
    ("oas_db", "customers", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_customers.json", None, "hourly", True, "v1", 0.9),
    ("oas_db", "policies", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_policies.json", None, "hourly", True, "v1", 0.9),
]

# ---------------------------------------
# STEP 4: Create DataFrame
# ---------------------------------------
control_df = spark.createDataFrame(records, schema=schema)

# ---------------------------------------
# STEP 5: Register table
# ---------------------------------------
control_df.createOrReplaceTempView("control_table")

# OUTPUT: show DataFrame and print final table
print("Control table DataFrame:")
control_df.show(truncate=False)

print("Final table from Spark SQL:")
spark.sql("SELECT * FROM control_table").show(truncate=False)



Control table DataFrame:
+-----------+------------------+-----------+------------------+-----------+----------------+--------------------+----------------------------------+---------------------+-------------+-----------+--------------+----------------------+
|source_name|entity_name       |source_type|source_path       |load_type  |watermark_column|last_processed_value|mapping_config_path               |transformation_module|schedule_type|active_flag|config_version|data_quality_threshold|
+-----------+------------------+-----------+------------------+-----------+----------------+--------------------+----------------------------------+---------------------+-------------+-----------+--------------+----------------------+
|carrier_crm|sales_transactions|csv        |/data/carrier_crm/|full       |NULL            |NULL                |/config/carrier_crm_sales.json    |NULL                 |daily        |true       |v1            |0.9                   |
|carrier_crm|customers         |csv

In [23]:
from pyspark.sql.functions import when, col

df = spark.table("control_table")

df = df.withColumn(
    "source_path",
    when(col("source_name") == "carrier_crm", "/content/carrier_crm_source.csv")
    .otherwise(col("source_path"))
)

df = df.withColumn(
    "mapping_config_path",
    when((col("source_name") == "carrier_crm") & (col("entity_name") == "customers"),
         "/content/carrier_crm_customers.json")
    .when((col("source_name") == "carrier_crm") & (col("entity_name") == "sales_transactions"),
         "/content/carrier_crm_sales.json")
    .when((col("source_name") == "carrier_crm") & (col("entity_name") == "policies"),
         "/content/carrier_crm_policies.json")
    .otherwise(col("mapping_config_path"))
)

df.createOrReplaceTempView("control_table")

In [24]:
"""Utilities for loading source runtime configuration from Delta control tables."""

from __future__ import annotations

import json
import logging
from typing import Any, Dict

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

LOGGER = logging.getLogger(__name__)


class ConfigNotFoundError(LookupError):
    """Raised when no active configuration exists for a source."""


class ConfigLoader:
    """Load active source configuration records from a Delta control table.

    Parameters
    ----------
    spark:
        Active :class:`SparkSession` used to query the control table.
    table_name:
        Fully-qualified Delta table name that stores source configurations.
    """

    def __init__(self, spark: SparkSession, table_name: str = "config.control_table") -> None:
        self.spark = spark
        self.table_name = table_name

    def load_config(self, source_name: str) -> Dict[str, Dict[str, Any]]:
        """Fetch all active source-entity configurations for a source.

        Parameters
        ----------
        source_name:
            Unique name of the data source whose active config is requested.

        Returns
        -------
        dict[str, dict[str, Any]]
            Mapping of ``entity_name`` to its active source configuration row.

        Raises
        ------
        ValueError
            If ``source_name`` is blank.
        ConfigNotFoundError
            If no active configuration record exists for ``source_name``.
        """

        normalized_source_name = source_name.strip() if source_name else ""
        if not normalized_source_name:
            raise ValueError("source_name must be a non-empty string.")

        config_df = self._build_active_config_query(normalized_source_name)
        config_rows = config_df.collect()
        if not config_rows:
            raise ConfigNotFoundError(
                f"No active configuration found in {self.table_name} for source_name='{normalized_source_name}'."
            )

        configs_by_entity: Dict[str, Dict[str, Any]] = {}
        for row in config_rows:
            config = row.asDict(recursive=True)
            entity_name = (config.get("entity_name") or "").strip()
            if not entity_name:
                LOGGER.warning(
                    "Skipping config row with blank entity_name for source_name='%s': %s",
                    normalized_source_name,
                    json.dumps(config, sort_keys=True, default=str),
                )
                continue

            if entity_name in configs_by_entity:
                LOGGER.warning(
                    "Duplicate active config detected for source_name='%s', entity_name='%s'. Keeping latest row.",
                    normalized_source_name,
                    entity_name,
                )

            configs_by_entity[entity_name] = config

        if not configs_by_entity:
            raise ConfigNotFoundError(
                f"No active entity-level configuration found in {self.table_name} for source_name='{normalized_source_name}'."
            )

        LOGGER.info(
            "Loaded %d active config rows for source_name='%s' from %s",
            len(configs_by_entity),
            normalized_source_name,
            self.table_name,
        )
        return configs_by_entity

    def _build_active_config_query(self, source_name: str) -> DataFrame:
        """Construct the active configuration query DataFrame for a source."""

        return (
            self.spark.table(self.table_name)
            .filter((F.col("source_name") == source_name) & (F.col("active_flag") == F.lit(True)))
        )


In [26]:
config = ConfigLoader(spark, "control_table")
carrier_config_dic = config.load_config("carrier_crm")

In [28]:
carrier_config_dic.values()

dict_values([{'source_name': 'carrier_crm', 'entity_name': 'sales_transactions', 'source_type': 'csv', 'source_path': '/content/carrier_crm_source.csv', 'load_type': 'full', 'watermark_column': None, 'last_processed_value': None, 'mapping_config_path': '/content/carrier_crm_sales.json', 'transformation_module': None, 'schedule_type': 'daily', 'active_flag': True, 'config_version': 'v1', 'data_quality_threshold': 0.9}, {'source_name': 'carrier_crm', 'entity_name': 'customers', 'source_type': 'csv', 'source_path': '/content/carrier_crm_source.csv', 'load_type': 'full', 'watermark_column': None, 'last_processed_value': None, 'mapping_config_path': '/content/carrier_crm_customers.json', 'transformation_module': None, 'schedule_type': 'daily', 'active_flag': True, 'config_version': 'v1', 'data_quality_threshold': 0.9}, {'source_name': 'carrier_crm', 'entity_name': 'policies', 'source_type': 'csv', 'source_path': '/content/carrier_crm_source.csv', 'load_type': 'full', 'watermark_column': Non